# Orislop AV-sync trainer

This notebook trains the lip/audio synchronization expert, calibrates an abstention band, tests on untouched identities, exports TorchScript, and can upload the result privately to Hugging Face. Select **Runtime → Change runtime type → GPU** before running it.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/coolguy860/Orislop-landing.git'
BRANCH = 'main'
REPO_DIR = Path('/content/orislop')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
TRAINER = REPO_DIR / 'training/orislop_avsync/train_avsync.py'
assert TRAINER.is_file(), f'Trainer not found: {TRAINER}. Push this branch or upload the training folder first.'
print('Repository:', REPO_DIR)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'training/orislop_avsync/requirements-colab.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
subprocess.run([sys.executable, str(TRAINER), 'self-test'], check=True)

## Mount Drive and configure the run

Your data should use `data/aligned/<speaker-id>/*.mp4` and `data/mismatched/<speaker-id>/*.mp4`. Optional `uncertain` and `not_applicable` folders audit the gates. Speaker folders prevent identity leakage across train/validation/test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/orislop-avsync')
DATA_ROOT = PROJECT / 'data'
MANIFEST = PROJECT / 'manifest.jsonl'
CACHE_DIR = PROJECT / 'cache'
PREPARED_MANIFEST = PROJECT / 'prepared.jsonl'
RUN_DIR = PROJECT / 'runs/avsync-v1'
EXPORT_DIR = PROJECT / 'exports/avsync-v1'
for label in ('aligned', 'mismatched', 'uncertain', 'not_applicable'):
    (DATA_ROOT / label).mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Add your clips under:', DATA_ROOT)

In [ ]:
def run_command(*parts):
    command = [str(part) for part in parts]
    print(' '.join(command))
    subprocess.run(command, check=True)

run_command(sys.executable, TRAINER, 'build-manifest',
            '--data-root', DATA_ROOT, '--output', MANIFEST)

## Preprocess

This extracts a 25 FPS mouth track and 16 kHz mono audio. Failed or faceless videos are reported next to the prepared manifest. Re-run with `--overwrite` only when preprocessing settings or source files change.

In [ ]:
run_command(sys.executable, TRAINER, 'prepare',
            '--manifest', MANIFEST, '--data-root', DATA_ROOT,
            '--cache-dir', CACHE_DIR, '--output-manifest', PREPARED_MANIFEST,
            '--max-seconds', '12')

## Train

Start with these settings. Reduce `BATCH_SIZE` if CUDA runs out of memory. Increase epochs only while validation ROC-AUC is still improving.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 16
WORKERS = 2
run_command(sys.executable, TRAINER, 'train',
            '--prepared-manifest', PREPARED_MANIFEST, '--output-dir', RUN_DIR,
            '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--workers', WORKERS,
            '--device', 'cuda')

## Calibrate on validation, then evaluate untouched test identities

The mismatch boundary targets 98% validation precision and leaves uncertain examples in an abstention band. The test cell reuses validation thresholds; it does not tune on test data.

In [ ]:
VALIDATION_JSON = RUN_DIR / 'validation.json'
run_command(sys.executable, TRAINER, 'evaluate',
            '--checkpoint', RUN_DIR / 'best.pt', '--prepared-manifest', PREPARED_MANIFEST,
            '--split', 'val', '--target-precision', '0.98', '--output', VALIDATION_JSON,
            '--batch-size', BATCH_SIZE, '--workers', WORKERS, '--device', 'cuda')

In [ ]:
TEST_JSON = RUN_DIR / 'test.json'
run_command(sys.executable, TRAINER, 'evaluate',
            '--checkpoint', RUN_DIR / 'best.pt', '--prepared-manifest', PREPARED_MANIFEST,
            '--split', 'test', '--thresholds-in', VALIDATION_JSON, '--output', TEST_JSON,
            '--batch-size', BATCH_SIZE, '--workers', WORKERS, '--device', 'cuda')

## Export

The export contains TorchScript plus a JSON contract with its hash, quality gates, temperature, and decision thresholds.

In [ ]:
run_command(sys.executable, TRAINER, 'export',
            '--checkpoint', RUN_DIR / 'best.pt', '--thresholds', VALIDATION_JSON,
            '--output-dir', EXPORT_DIR)
print('Export:', EXPORT_DIR)

## Test one video

Change `EXAMPLE_VIDEO`. Closed lips, missing speech, or no usable face should return `not_applicable` without calling the neural model.

In [ ]:
EXAMPLE_VIDEO = next((DATA_ROOT / 'aligned').rglob('*.mp4'))
run_command(sys.executable, TRAINER, 'predict',
            '--video', EXAMPLE_VIDEO, '--checkpoint', RUN_DIR / 'best.pt',
            '--thresholds', VALIDATION_JSON, '--device', 'cuda')

## Optional private Hugging Face upload

Create an `HF_TOKEN` entry under Colab's key icon. Do not paste tokens into the notebook. Review licenses and test results before making the repository public.

In [ ]:
UPLOAD_TO_HF = False
HF_REPO_ID = 'gonnerthetooner/orislop-avsync'
if UPLOAD_TO_HF:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    run_command(sys.executable, TRAINER, 'export',
                '--checkpoint', RUN_DIR / 'best.pt', '--thresholds', VALIDATION_JSON,
                '--output-dir', EXPORT_DIR, '--push-to-hub', '--hf-repo-id', HF_REPO_ID, '--private')
else:
    print('Upload disabled. Set UPLOAD_TO_HF = True when ready.')